In [0]:
create or replace table production.supply_analytics.feature_audit_test as

WITH base_tours AS (
  SELECT DISTINCT
    dt.tour_id,
    o.tour_option_id,
    dt.tour_title,
    o.tour_option_title,
    dt.user_id AS supplier_id,
    dtl.location_name,
    dt.activity_type,
    dt.category AS tour_category,
    dt.date_of_creation,
    dt.date_of_first_online
  FROM production.dwh.dim_tour dt
  LEFT JOIN production.dwh.dim_tour_option o ON dt.tour_id = o.tour_id
  LEFT JOIN production.dwh.dim_tour_location dtl ON dtl.location_id = dt.location_id AND dtl.tour_id = dt.tour_id
  WHERE dt.gyg_status = 'active'
    AND dt.status = 'active'
    AND dt.is_online = true
    AND o.status = 'active'
),

pricing_config_raw AS (
  SELECT  
    tpc.tour_id,
    tpc.tour_option_id,
    tpc.ticket_category,
    tpc.price_scale_max_participants,
    tpc.price_scale_min_participants,
    tpc.price_eur
  FROM production.dwh.dim_tour_pricing_configuration tpc
  WHERE tpc.tour_pricing_status = 'active'
),

live_pricing_flags AS (
  SELECT DISTINCT
    tpc.tour_option_id,
    MAX(CASE WHEN poa.pricing_engine IS NOT NULL THEN poa.pricing_engine END) AS pricing_engine,
    
    -- Individual API flags
    MAX(CASE WHEN ins.uses_prices_over_api = true THEN 1 ELSE 0 END) AS uses_api_prices,
    MAX(CASE WHEN ins.uses_pricing_scales_over_api = true THEN 1 ELSE 0 END) AS uses_api_scales,
    MAX(CASE WHEN ins.uses_live_availability_and_price = true THEN 1 ELSE 0 END) AS uses_live_availability,
    
    -- Config table flags
    MAX(CASE WHEN tpc.price_scale_max_participants > 1 THEN 1 ELSE 0 END) AS has_config_scales,
    MAX(CASE WHEN tpc.price_eur > 0 THEN 1 ELSE 0 END) AS has_config_prices,
    
    -- Primary pricing mechanism (mutually exclusive hierarchy)
    CASE 
      WHEN MAX(CASE WHEN ins.uses_pricing_scales_over_api = true THEN 1 ELSE 0 END) = 1 
        THEN 'API_SCALES'
      WHEN MAX(CASE WHEN ins.uses_prices_over_api = true THEN 1 ELSE 0 END) = 1 
        THEN 'API_PRICES'
      WHEN MAX(CASE WHEN tpc.price_scale_max_participants > 1 THEN 1 ELSE 0 END) = 1
        THEN 'CONFIG_SCALES'
      WHEN MAX(CASE WHEN tpc.price_eur > 0 THEN 1 ELSE 0 END) = 1
        THEN 'CONFIG_PRICES'
      ELSE 'NO_PRICING'
    END AS pricing_mechanism,
    
    -- Simplified binary flags
    CASE 
      WHEN MAX(CASE WHEN ins.uses_pricing_scales_over_api = true OR ins.uses_prices_over_api = true THEN 1 ELSE 0 END) = 1 
        THEN 1 
      ELSE 0 
    END AS uses_any_api_pricing,
    
    CASE 
      WHEN MAX(CASE WHEN ins.uses_pricing_scales_over_api = true THEN 1 ELSE 0 END) = 1 
        OR MAX(CASE WHEN tpc.price_scale_max_participants > 1 THEN 1 ELSE 0 END) = 1
        THEN 1 
      ELSE 0 
    END AS uses_any_scale_pricing
    
  FROM production.dwh.dim_tour_pricing_configuration tpc
  LEFT JOIN production.connectivity.price_over_api_activation_inventory_view poa 
    ON poa.tour_option_id = tpc.tour_option_id 
  LEFT JOIN production.db_mirror_dbz.inventory__inventory_settings ins 
    ON ins.tour_option_id = tpc.tour_option_id
  WHERE tpc.tour_pricing_status = 'active'
  GROUP BY tpc.tour_option_id
),

valid_customers AS (
  SELECT DISTINCT fb.customer_id_anon
  FROM production.dwh.fact_booking fb
  LEFT JOIN production.dwh.dim_customer_ticket_reseller reseller
    ON reseller.customer_id_anon = fb.customer_id_anon
  WHERE fb.date_of_checkout >= CURRENT_DATE - INTERVAL 12 MONTH
    AND fb.status_id = 1
    AND reseller.customer_id_anon IS NULL
),

performance AS (
  SELECT 
    fb.tour_id,
    fb.tour_option_id,
    COUNT(DISTINCT fb.booking_id) AS bookings_12mo,
    SUM(fb.tickets) AS tickets_12mo,
    SUM(fb.gmv) AS gmv_12mo,
    SUM(fb.nr) AS nr_12mo,
    COUNT(DISTINCT fb.customer_id_anon) AS unique_customers_12mo
  FROM production.dwh.fact_booking fb
  INNER JOIN valid_customers vc 
    ON fb.customer_id_anon = vc.customer_id_anon
  WHERE fb.date_of_checkout >= CURRENT_DATE - INTERVAL 12 MONTH
    AND fb.status_id = 1
  GROUP BY fb.tour_id, fb.tour_option_id
),

category_features AS (
  SELECT 
    tour_id,
    tour_option_id,
    COUNT(DISTINCT ticket_category) AS total_categories,
    COUNT(DISTINCT NULLIF(ticket_category, '')) AS total_non_null_categories,
    MAX(CASE WHEN ticket_category = 'adult' THEN 1 ELSE 0 END) AS has_adult,
    MAX(CASE WHEN ticket_category = 'child' THEN 1 ELSE 0 END) AS has_child,
    MAX(CASE WHEN ticket_category = 'infant' THEN 1 ELSE 0 END) AS has_infant,
    MAX(CASE WHEN ticket_category = 'senior' THEN 1 ELSE 0 END) AS has_senior,
    MAX(CASE WHEN ticket_category = 'student' THEN 1 ELSE 0 END) AS has_student,
    MAX(CASE WHEN ticket_category = 'youth' THEN 1 ELSE 0 END) AS has_youth,
    MAX(CASE WHEN ticket_category = 'military' THEN 1 ELSE 0 END) AS has_military,
    MAX(CASE WHEN ticket_category = 'eu_citizen' THEN 1 ELSE 0 END) AS has_eu_citizen,
    MAX(CASE WHEN ticket_category = 'eu_citizen_student' THEN 1 ELSE 0 END) AS has_eu_citizen_student,
    (MAX(CASE WHEN ticket_category = 'adult' THEN 1 ELSE 0 END) +
     MAX(CASE WHEN ticket_category = 'child' THEN 1 ELSE 0 END) +
     MAX(CASE WHEN ticket_category = 'infant' THEN 1 ELSE 0 END) +
     MAX(CASE WHEN ticket_category = 'senior' THEN 1 ELSE 0 END) +
     MAX(CASE WHEN ticket_category = 'student' THEN 1 ELSE 0 END) +
     MAX(CASE WHEN ticket_category = 'youth' THEN 1 ELSE 0 END) +
     MAX(CASE WHEN ticket_category = 'military' THEN 1 ELSE 0 END) +
     MAX(CASE WHEN ticket_category = 'eu_citizen' THEN 1 ELSE 0 END) +
     MAX(CASE WHEN ticket_category = 'eu_citizen_student' THEN 1 ELSE 0 END)) AS category_breadth_score,
    MAX(CASE WHEN ticket_category = 'adult' AND price_scale_min_participants = 1 
             THEN price_eur END) AS adult_base_price,
    MAX(CASE WHEN ticket_category = 'child' AND price_scale_min_participants = 1 
             THEN price_eur END) AS child_base_price,
    MAX(CASE WHEN ticket_category = 'senior' AND price_scale_min_participants = 1 
             THEN price_eur END) AS senior_base_price,
    MAX(CASE WHEN ticket_category = 'student' AND price_scale_min_participants = 1 
             THEN price_eur END) AS student_base_price
  FROM pricing_config_raw
  GROUP BY tour_id, tour_option_id
),

scale_features AS (
  SELECT 
    tour_id,
    tour_option_id,
    COUNT(DISTINCT CONCAT(price_scale_min_participants, '-', price_scale_max_participants)) AS total_unique_scales,
    MIN(price_scale_min_participants) AS min_group_size,
    MAX(price_scale_max_participants) AS max_group_size,
    COUNT(DISTINCT CASE WHEN ticket_category = 'adult' 
                        THEN CONCAT(price_scale_min_participants, '-', price_scale_max_participants) 
                   END) AS adult_scale_tiers,
    MAX(CASE WHEN ticket_category = 'adult' THEN price_eur END) AS adult_max_price,
    MIN(CASE WHEN ticket_category = 'adult' AND price_eur > 0 THEN price_eur END) AS adult_min_price,
    (MAX(CASE WHEN ticket_category = 'adult' THEN price_eur END) - 
     MIN(CASE WHEN ticket_category = 'adult' AND price_eur > 0 THEN price_eur END)) / 
     NULLIF(MAX(CASE WHEN ticket_category = 'adult' THEN price_eur END), 0) * 100 AS adult_max_discount_pct,
    MAX(CASE WHEN price_scale_max_participants > 1 THEN 1 ELSE 0 END) AS has_scale_pricing,
    COUNT(*) / NULLIF(COUNT(DISTINCT ticket_category), 0) AS avg_tiers_per_category
  FROM pricing_config_raw
  WHERE price_eur > 0
  GROUP BY tour_id, tour_option_id
),

option_level AS (
  SELECT 
    t.tour_id,
    t.tour_title, 
    t.tour_option_id,
    t.supplier_id,
    t.location_name,
    t.activity_type,
    t.tour_category,
    DATEDIFF(day, t.date_of_first_online, CURRENT_DATE) AS days_since_first_online,
    
    -- Category features
    COALESCE(cf.total_non_null_categories, 0) AS total_categories,
    COALESCE(cf.category_breadth_score, 0) AS category_breadth,
    cf.has_adult,
    cf.has_child,
    cf.has_senior,
    cf.has_student,
    cf.has_youth,
    cf.has_military,
    cf.has_eu_citizen,
    
    -- Scale features
    COALESCE(sf.total_unique_scales, 0) AS total_group_scales,
    COALESCE(sf.adult_scale_tiers, 0) AS adult_scale_tiers,
    COALESCE(sf.max_group_size, 0) AS max_capacity,
    COALESCE(sf.adult_max_discount_pct, 0) AS volume_discount_pct,
    COALESCE(sf.has_scale_pricing, 0) AS has_scale_pricing,
    COALESCE(sf.avg_tiers_per_category, 0) AS avg_tiers_per_category,
    
    -- Detailed pricing mechanism flags
    lpf.pricing_mechanism,
    COALESCE(lpf.uses_api_prices, 0) AS uses_api_prices,
    COALESCE(lpf.uses_api_scales, 0) AS uses_api_scales,
    COALESCE(lpf.has_config_scales, 0) AS has_config_scales,
    COALESCE(lpf.has_config_prices, 0) AS has_config_prices,
    COALESCE(lpf.uses_any_api_pricing, 0) AS uses_any_api_pricing,
    COALESCE(lpf.uses_any_scale_pricing, 0) AS uses_any_scale_pricing,
    COALESCE(lpf.uses_live_availability, 0) AS uses_live_availability,
    lpf.pricing_engine,
    
    -- Performance
    COALESCE(p.bookings_12mo, 0) AS bookings_12mo,
    COALESCE(p.tickets_12mo, 0) AS tickets_12mo,
    COALESCE(p.gmv_12mo, 0) AS gmv_12mo,
    COALESCE(p.nr_12mo, 0) AS nr_12mo,
    COALESCE(p.unique_customers_12mo, 0) AS unique_customers_12mo,
    
    -- Calculated metrics
    p.gmv_12mo / NULLIF(p.bookings_12mo, 0) AS avg_booking_value,
    p.tickets_12mo / NULLIF(p.bookings_12mo, 0) AS avg_group_size,
    p.nr_12mo / NULLIF(p.gmv_12mo, 0) * 100 AS nr_margin_pct,
    p.bookings_12mo / NULLIF(DATEDIFF(day, t.date_of_first_online, CURRENT_DATE), 0) * 365 AS bookings_per_year
    
  FROM base_tours t
  LEFT JOIN category_features cf USING(tour_id, tour_option_id)
  LEFT JOIN scale_features sf USING(tour_id, tour_option_id)
  LEFT JOIN live_pricing_flags lpf USING(tour_option_id)
  LEFT JOIN performance p USING(tour_id, tour_option_id)
),

tour_level AS (
  SELECT 
    tour_id,
    tour_title,
    supplier_id,
    location_name,
    activity_type,
    tour_category,
    MAX(days_since_first_online) AS days_since_first_online,
    COUNT(DISTINCT tour_option_id) AS total_options,
    COUNT(DISTINCT CASE WHEN bookings_12mo > 0 THEN tour_option_id END) AS options_with_bookings,
    
    -- Category features - CORRECTED: Count exact distinct categories
    (MAX(has_adult) + MAX(has_child) + MAX(has_senior) + MAX(has_student) + 
     MAX(has_youth) + MAX(has_military) + MAX(has_eu_citizen)) AS total_distinct_categories,
    MAX(category_breadth) AS max_category_breadth,
    MAX(has_adult) AS tour_has_adult,
    MAX(has_child) AS tour_has_child,
    MAX(has_student) AS tour_has_student,
    MAX(has_senior) AS tour_has_senior,
    MAX(has_youth) AS tour_has_youth,
    MAX(has_military) AS tour_has_military,
    MAX(has_eu_citizen) AS tour_has_eu_citizen,
    
    -- Scale features
    AVG(total_group_scales) AS avg_scales_per_option,
    AVG(adult_scale_tiers) AS avg_adult_scale_tiers,
    MAX(max_capacity) AS max_group_capacity,
    AVG(volume_discount_pct) AS avg_volume_discount_pct,
    SUM(has_scale_pricing) AS options_with_scale_pricing,
    AVG(avg_tiers_per_category) AS avg_tiers_per_category,
    
    -- Pricing mechanism distribution
    SUM(uses_api_prices) AS options_with_api_prices,
    SUM(uses_api_scales) AS options_with_api_scales,
    SUM(has_config_scales) AS options_with_config_scales,
    SUM(has_config_prices) AS options_with_config_prices,
    SUM(uses_any_api_pricing) AS options_with_any_api,
    SUM(uses_any_scale_pricing) AS options_with_any_scales,
    SUM(uses_live_availability) AS options_with_live_availability,
    
    -- Pricing mechanism breakdown (counts)
    SUM(CASE WHEN pricing_mechanism = 'API_SCALES' THEN 1 ELSE 0 END) AS options_api_scales,
    SUM(CASE WHEN pricing_mechanism = 'API_PRICES' THEN 1 ELSE 0 END) AS options_api_prices,
    SUM(CASE WHEN pricing_mechanism = 'CONFIG_SCALES' THEN 1 ELSE 0 END) AS options_config_scales,
    SUM(CASE WHEN pricing_mechanism = 'CONFIG_PRICES' THEN 1 ELSE 0 END) AS options_config_prices,
    SUM(CASE WHEN pricing_mechanism = 'NO_PRICING' THEN 1 ELSE 0 END) AS options_no_pricing,
    
    -- Primary pricing mechanism for tour (mode)
    MODE(pricing_mechanism) AS tour_primary_pricing_mechanism,
    
    -- Performance
    SUM(bookings_12mo) AS total_bookings,
    SUM(tickets_12mo) AS total_tickets,
    SUM(gmv_12mo) AS total_gmv,
    SUM(nr_12mo) AS total_nr,
    SUM(unique_customers_12mo) AS total_unique_customers,
    
    -- Averages
    AVG(avg_booking_value) AS avg_booking_value,
    AVG(avg_group_size) AS avg_group_size,
    AVG(nr_margin_pct) AS avg_nr_margin_pct,
    AVG(bookings_per_year) AS avg_bookings_per_year
    
  FROM option_level
  GROUP BY tour_id, tour_title, supplier_id, location_name, activity_type, tour_category
)

SELECT 
  *,
  
  -- CORRECTED: Feature adoption score using exact category count
  (CASE WHEN total_distinct_categories >= 2 THEN 1 ELSE 0 END +
   CASE WHEN options_with_any_scales > 0 THEN 1 ELSE 0 END +
   CASE WHEN total_options > 1 AND options_with_bookings > 1 THEN 1 ELSE 0 END +
   CASE WHEN options_with_any_api > 0 THEN 1 ELSE 0 END) AS pricing_feature_score,
   
  -- Performance tier
  CASE 
    WHEN total_gmv = 0 OR total_gmv IS NULL THEN 'No Revenue'
    WHEN NTILE(10) OVER (ORDER BY CASE WHEN total_gmv > 0 THEN total_gmv END) >= 9 
    THEN 'Top 10%'
    WHEN NTILE(10) OVER (ORDER BY CASE WHEN total_gmv > 0 THEN total_gmv END) >= 7 
    THEN 'Top 25%'
    WHEN NTILE(10) OVER (ORDER BY CASE WHEN total_gmv > 0 THEN total_gmv END) >= 5 
    THEN 'Top 50%'
    WHEN NTILE(10) OVER (ORDER BY CASE WHEN total_gmv > 0 THEN total_gmv END) >= 3 
    THEN 'Top 75%'
    ELSE 'Bottom 25%'
  END AS performance_tier,
  
  CASE 
    WHEN total_gmv = 0 OR total_gmv IS NULL THEN 0
    ELSE NTILE(10) OVER (ORDER BY CASE WHEN total_gmv > 0 THEN total_gmv END)
  END AS performance_decile,
  
  -- CORRECTED: Pricing sophistication using exact category count
  CASE 
    WHEN (CASE WHEN total_distinct_categories >= 2 THEN 1 ELSE 0 END +
          CASE WHEN options_with_any_scales > 0 THEN 1 ELSE 0 END +
          CASE WHEN total_options > 1 AND options_with_bookings > 1 THEN 1 ELSE 0 END +
          CASE WHEN options_with_any_api > 0 THEN 1 ELSE 0 END) >= 3 
    THEN 'Advanced Pricing'
    WHEN (CASE WHEN total_distinct_categories >= 2 THEN 1 ELSE 0 END +
          CASE WHEN options_with_any_scales > 0 THEN 1 ELSE 0 END +
          CASE WHEN total_options > 1 AND options_with_bookings > 1 THEN 1 ELSE 0 END +
          CASE WHEN options_with_any_api > 0 THEN 1 ELSE 0 END) = 2 
    THEN 'Intermediate Pricing'
    WHEN (CASE WHEN total_distinct_categories >= 2 THEN 1 ELSE 0 END +
          CASE WHEN options_with_any_scales > 0 THEN 1 ELSE 0 END +
          CASE WHEN total_options > 1 AND options_with_bookings > 1 THEN 1 ELSE 0 END +
          CASE WHEN options_with_any_api > 0 THEN 1 ELSE 0 END) = 1 
    THEN 'Basic Pricing'
    ELSE 'Minimal Pricing'
  END AS pricing_sophistication,
  
  -- Pricing mechanism mix
  ROUND(options_with_any_api * 100.0 / NULLIF(total_options, 0), 1) AS pct_options_using_api,
  ROUND(options_with_any_scales * 100.0 / NULLIF(total_options, 0), 1) AS pct_options_using_scales,
  ROUND(options_api_scales * 100.0 / NULLIF(total_options, 0), 1) AS pct_options_api_scales,
  ROUND(options_config_scales * 100.0 / NULLIF(total_options, 0), 1) AS pct_options_config_scales

FROM tour_level
ORDER BY total_gmv DESC NULLS LAST;
